# Plots for Margins

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
os.chdir('../..')
sys.path.insert(1, os.path.join(sys.path[0], '../..'))

In [ ]:
import logging
logging.getLogger('fontTools').setLevel(logging.ERROR)
logging.getLogger('matplotlib').setLevel(logging.ERROR)

In [ ]:
from notebooks.articles.utils import (
    evaluate_sv,
    plot_inter_class_similarity,
    plot_intra_class_similarity,
    plot_inter_speaker_center_similarity
)

from notebooks.evaluation.sv_visualization import (
    det_curve,
    scores_distribution,
)

from plotnine import *
import patchworklib as pw

import numpy as np
import pandas as pd

import json

In [ ]:
from dataclasses import dataclass
from typing import List, Dict

import torch


@dataclass
class Model:

    scores: List[float] = None
    targets: List[int] = None
    embeddings: Dict[str, torch.Tensor] = None


def get_models_for_visualization(scores, names=None):
    if names is None:
        names = list(scores.keys())

    models = {
        k:Model(v['scores'], v['targets'])
        for k, v
        in scores.items()
        if k in names
    }

    return models

In [ ]:
MODELS = {
    "Baseline":        "models/ssl/voxceleb2/simclr/simclr_proj-none_t-0.03/",

    "CosFace (m=0.05)":  "models/margins/voxceleb2/simclr/cosface_m-0.05/",
    "CosFace (m=0.1)":   "models/margins/voxceleb2/simclr/cosface_m-0.1/",
    "CosFace (m=0.2)":   "models/margins/voxceleb2/simclr/cosface_m-0.2/",
    "CosFace (m=0.3)":   "models/margins/voxceleb2/simclr/cosface_m-0.3/",
    "CosFace (m=0.5)":   "models/margins/voxceleb2/simclr/cosface_m-0.5/",

    "AdaFace (m=0.05)":  "models/margins/voxceleb2/simclr/adaface_m-0.05/",
    "AdaFace (m=0.1)":   "models/margins/voxceleb2/simclr/adaface_m-0.1/",
    "AdaFace (m=0.2)":   "models/margins/voxceleb2/simclr/adaface_m-0.2/",
    "AdaFace (m=0.3)":   "models/margins/voxceleb2/simclr/adaface_m-0.3/",
    "AdaFace (m=0.5)":   "models/margins/voxceleb2/simclr/adaface_m-0.5/",
}

## Palette

In [ ]:
from mizani.palettes import hue_pal
import matplotlib.pyplot as plt

palette = hue_pal(h=0.01, l=0.6, s=0.65, color_space="hls")(len(MODELS.keys()))

plt.figure(figsize=(8, 1))
for i, color in enumerate(palette):
    plt.bar(i, 1, color=color)
plt.xticks(range(len(MODELS.keys())), MODELS.keys())
plt.show()

palette = dict(zip(MODELS.keys(), palette))
print(palette)

In [ ]:
MODELS_ORDER = list(MODELS.keys())
MODELS_PALETTE = palette

MODELS_ORDER, MODELS_PALETTE

In [ ]:
MODELS_COSFACE = {k:v for k, v in MODELS.items() if 'CosFace' in k or 'Baseline' in k}
MODELS_COSFACE_ORDER = list(MODELS_COSFACE.keys())

In [ ]:
MODELS_ADAFACE = {k:v for k, v in MODELS.items() if 'AdaFace' in k or 'Baseline' in k}
MODELS_ADAFACE_ORDER = list(MODELS_ADAFACE.keys())

## Metrics

In [ ]:
vox1o_scores = evaluate_sv(MODELS, 'embeddings_vox1_avg.pt', trials=["voxceleb1_test_O"])

In [ ]:
vox1h_scores = evaluate_sv(MODELS, 'embeddings_vox1_avg.pt', trials=["voxceleb1_test_H"])

## DET

In [ ]:
p = det_curve(get_models_for_visualization(vox1o_scores))
p += scale_color_manual(values=MODELS_PALETTE, limits=MODELS_ORDER)
# p.save('det.pdf')
p

In [ ]:
p = det_curve(get_models_for_visualization(vox1h_scores))
p += scale_color_manual(values=MODELS_PALETTE, limits=MODELS_ORDER)
# p.save('det.pdf')
p

## Convergence

In [ ]:
res = []
for name, path in MODELS_COSFACE.items():
    try:
        with open(f'{path}/training.json', "r") as f:
            train = json.load(f)
    except:
        continue
    for epoch, metrics in train.items():
        res.append({'Epoch': int(epoch), 'Model': name, **metrics})

data = pd.DataFrame(res)

def create_plot(y, label):
    p = (
        ggplot(data, aes(x='Epoch', y=y, color='factor(Model)'))
        + geom_line(size=1)
        # + geom_point()
        + scale_color_manual(values=MODELS_PALETTE, limits=MODELS_COSFACE_ORDER)
        + labs(x='Epoch', y=label, color='Models')
        + theme_bw()
        + theme(
            figure_size=(8, 4.75),
            text=element_text(size=14),
            legend_position='top',
            legend_title=element_blank(),
            legend_key_spacing_x=15
        )
        + guides(color=guide_legend(nrow=1))
    )
    return p


g_loss = create_plot('train/loss', 'Train loss')
g_eer = create_plot('val/sv_cosine/voxceleb1_test_O/eer', 'EER (%)')
g_mindcf = create_plot('val/sv_cosine/voxceleb1_test_O/mindcf', 'minDCF (p=0.01)')

p = g_eer

# p.save("convergence.pdf")
p

## Intra/inter-speaker similarity

In [ ]:
MODELS_PALETTE_ALPHA = {k:v + "B3" for k, v in MODELS_PALETTE.items()}
MODELS_PALETTE_ALPHA

In [ ]:
MODELS_COSFACE_ = {k:f'{v}/embeddings_vox1_avg.pt' for k, v in MODELS_COSFACE.items()}
MODELS_ADAFACE_ = {k:f'{v}/embeddings_vox1_avg.pt' for k, v in MODELS_ADAFACE.items()}

In [ ]:
def get_short_names(models):
    res = {}
    for k in models.keys():
        if 'm=' in k:
            res[k] = f"{k.split('(m=')[1].split(')')[0]}"
        else:
            res[k] = k
    return res

In [ ]:
p, stats = plot_intra_class_similarity('speaker', MODELS_COSFACE_)
p += scale_x_discrete(labels=get_short_names(MODELS_COSFACE_))
p += scale_fill_manual(values=MODELS_PALETTE_ALPHA, limits=MODELS_COSFACE_ORDER)
p += ggtitle("CosFace")
p += theme(text=element_text(size=18), plot_title=element_text(size=20))
p.save("intra_cosface_vox1.pdf")
p

In [ ]:
stats

In [ ]:
p, stats = plot_intra_class_similarity('speaker', MODELS_ADAFACE_)
p += scale_x_discrete(labels=get_short_names(MODELS_ADAFACE_))
p += scale_fill_manual(values=MODELS_PALETTE_ALPHA, limits=MODELS_ADAFACE_ORDER)
p += ggtitle("AdaFace")
p += theme(text=element_text(size=18), plot_title=element_text(size=20))
p.save("intra_adaface_vox1.pdf")
p

In [ ]:
stats

In [ ]:
p, stats = plot_inter_class_similarity('speaker', MODELS_COSFACE_, nb_samples=100)
p += scale_x_discrete(labels=get_short_names(MODELS_COSFACE_))
p += scale_fill_manual(values=MODELS_PALETTE_ALPHA, limits=MODELS_COSFACE_ORDER)
p += ggtitle("CosFace")
p += theme(text=element_text(size=18), plot_title=element_text(size=20))
p.save("inter_cosface_vox1.pdf")
p

In [ ]:
stats

In [ ]:
p, stats = plot_inter_class_similarity('speaker', MODELS_ADAFACE_, nb_samples=100)
p += scale_x_discrete(labels=get_short_names(MODELS_ADAFACE_))
p += scale_fill_manual(values=MODELS_PALETTE_ALPHA, limits=MODELS_ADAFACE_ORDER)
p += ggtitle("AdaFace")
p += theme(text=element_text(size=18), plot_title=element_text(size=20))
p.save("inter_adaface_vox1.pdf")
p

In [ ]:
stats

## Score distribution

In [ ]:
scores_distribution(get_models_for_visualization(vox1o_scores, [
    "Baseline",
    "CosFace (m=0.1)",
    "CosFace (m=0.2)",
    "CosFace (m=0.3)",
]), use_angle=False)

In [ ]:
p = scores_distribution(get_models_for_visualization(vox1h_scores, [
    "Baseline",
    "CosFace (m=0.1)",
    "CosFace (m=0.2)",
    "CosFace (m=0.3)",
]), use_angle=False)

p.save("score_distribution_cosface_vox1h.pdf")

p

In [ ]:
p = scores_distribution(get_models_for_visualization(vox1h_scores, [
    "Baseline",
    "AdaFace (m=0.1)",
    "AdaFace (m=0.2)",
    "AdaFace (m=0.3)",
]), use_angle=False)

p.save("score_distribution_adaface_vox1h.pdf")

p